# 05 – Tren en lineær klassifikator

**Tid:** 90 minutter. **Mål:** samle bildedata, modell, tap og optimizer i en
synlig treningsløkke. Nå skal modellen faktisk lære å gjenkjenne sifre.

Denne notatboken har sin egen kjerne. Du skal gjenbruke modellideen fra
notatbok 03, ikke variablene fra den gamle kjernen.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Før du begynner: forklar treningen til en partner

Bruk fem minutter uten kode: Hvilke fem handlinger inngår i én treningsrunde?
Hva er formen på bildene, skårene og etikettene? Når endres vektene?
Sjekk mot notatbok 04. Nå bruker vi samme mønster på MNIST i stedet for to små vektorer.


In [ ]:
# Formene vi forventer i hver batch (B kan være mindre enn 64):
# bilder [B, 1, 28, 28] -> skårer [B, 10]
# etiketter [B], int64 -> tap [], ett tall


## Grupper til trening og evaluering

Som i notatbok 03 oppretter vi datalasterne direkte. Treningsbildene
stokkes. Testbildene brukes bare til evaluering og beholder rekkefølgen.
Ingen global frøsetting er nødvendig; målingene kan variere litt.


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)


## Oppgave 1: Sett opp MNIST-modellen

Lag `model` med `nn.Flatten()` og `nn.Linear(784, 10)`, som i notatbok 03.
Lag `loss_fn = nn.CrossEntropyLoss()` og `optimizer` med
`torch.optim.SGD`, modellens parametere og læringsrate 0.1.

<details>
<summary>Hint</summary>

Optimizer må få `model.parameters()` fra akkurat den modellen du trener.

</details>


In [ ]:
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
print(model)


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert model(torch.zeros(2, 1, 28, 28)).shape == (2, 10)


## Fra ett steg til flere epoker

Én **epoke** er én gjennomgang av alle treningsbildene. Datalasteren gir én
gruppe om gangen. Den siste gruppen kan være mindre, så bruk `len(images)`
i beregningene. `CrossEntropyLoss` gir gjennomsnittstap for gruppen.
For et riktig epokegjennomsnitt vekter vi derfor gruppetapet med gruppestørrelsen.

`model.train()` setter modellen i treningsmodus. Det er en god vane, selv om
akkurat vår lineære modell oppfører seg likt i trenings- og evalueringsmodus.


## Oppgave 2: Skriv treningsløkken

Tren modellen i fem epoker. Opprett `training_loss = []`.
For hver epoke:

1. Sett modellen i treningsmodus og start en tapsum på 0.
2. Gå gjennom `train_loader` med `for images, labels in train_loader`.
3. Nullstill gradientene, beregn skårer og tap, kall `backward()` og `step()`.
4. Legg `loss.item() * len(images)` til tapsummen.
5. Del tapsummen på antall treningsbilder og lagre resultatet i `training_loss`.

Skriv ut gjennomsnittstapet etter hver epoke. Kjør modellcellen over på nytt
hvis du vil starte treningen fra begynnelsen.

<details>
<summary>Hint</summary>

Bruk treningsmønsteret fra det lille eksemplet. Bare modell, data og tapsfunksjon er annerledes.

</details>

Før du kjører fem epoker: sett midlertidig antallet til én. Kontroller at tapet
er endelig og at løkken fullføres. Opprett deretter modell og optimizer på nytt
og kjør fem epoker. Flere kjøringer av bare treningscellen fortsetter treningen.


In [ ]:
training_loss = []
for epoch in range(5):
    model.train()
    loss_sum = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(images)
    mean_loss = loss_sum / len(train_data)
    training_loss.append(mean_loss)
    print(f"Epoke {epoch + 1}: treningstap {mean_loss:.4f}")


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert len(training_loss) == 5
assert all(np.isfinite(training_loss))
assert training_loss[-1] < training_loss[0]


### Sjekk forståelsen

Oppgi første og siste treningstap. Hva antyder utviklingen?
Hva er forskjellen på én gruppe, én parameteroppdatering og én epoke?
Hvorfor ganger vi gruppetapet med antall bilder før vi summerer?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.

Med 6 000 bilder og batchstørrelse 64: hvor mange oppdateringer blir det
i én epoke, og hvor mange bilder er det i siste batch?


**Mitt svar:**

Et eksempel er at tapet faller fra omtrent 1,01 til 0,41. Tallene
varierer med tilfeldig initialisering og stokking; oppgi dine egne.
Lavere tap betyr bedre tilpasning til treningsdata etter tapsfunksjonens
mål, men beviser ikke alene at modellen generaliserer.
Én gruppe inneholder flere bilder og gir én oppdatering i denne løkken.
Én epoke omfatter alle gruppene. Vi vekter etter antall bilder, slik
at en liten siste gruppe ikke teller like mye som en full gruppe.

Det blir 94 oppdateringer per epoke: 93 fulle grupper og én gruppe med 48 bilder.


## Eksempel: nøyaktighet fra skårer

`argmax(dim=1)` velger klasse per eksempel. Sammenligningen med etikettene
gir boolske verdier, som kan telles med `sum()`.
Ved evaluering bruker vi `model.eval()` og `torch.no_grad()`:
førstnevnte velger evalueringsmodus, sistnevnte slår av gradientregistrering.
Vi verken beregner gradienter eller oppdaterer parametere på testdata.


In [ ]:
example_scores = torch.tensor([[3., 0., 1.], [0., 1., 4.], [0., 2., 1.]])
example_labels = torch.tensor([0, 1, 1])
predictions = example_scores.argmax(dim=1)
print("Prediksjoner:", predictions)
print("Nøyaktighet:", (predictions == example_labels).float().mean().item())


## Oppgave 3: Evaluer på testbilder

Sett modellen i evalueringsmodus. Under `torch.no_grad()`: gå gjennom
`test_loader`, beregn skårer, velg prediksjoner og tell riktige.
Lag `test_accuracy` som antall riktige delt på antall testbilder.
Skriv ut resultatet som prosent. Vis også `training_loss` som en kurve.

<details>
<summary>Hint</summary>

Start med `num_correct = 0`. Bruk `(predictions == labels).sum().item()`.

</details>

Rapporter også `num_correct` av `len(test_data)`; sammenlign med en grov
referanse på 10 % for tilfeldig gjetting blant ti klasser.


In [ ]:
model.eval()
num_correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        scores = model(images)
        predictions = scores.argmax(dim=1)
        num_correct += (predictions == labels).sum().item()
test_accuracy = num_correct / len(test_data)
print(f"Testnøyaktighet: {test_accuracy:.1%} ({num_correct}/{len(test_data)})")
fig, ax = plt.subplots()
ax.plot(range(1, 6), training_loss, marker="o")
ax.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="Den lineære modellen lærer")
plt.show()


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert 0 <= test_accuracy <= 1

assert test_accuracy == num_correct / len(test_data)


### Sjekk forståelsen

Hva ble testnøyaktigheten, og hva betyr den i antall riktige av 1 000 bilder?
Hva ville vært galt med å kalle `optimizer.step()` i testløkken?
Er `eval()` og `no_grad()` det samme? Forklar forskjellen.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Et eksempel er 91,4 %, altså 914 riktige av 1 000 bilder. Det er ikke
en fasit: uten fast frø varierer modellen og resultatet. Bruk din egen
målte prosent og antall riktige. Oppdateringer på testdata ville gjort
testen del av treningen. `eval()` velger modellmodus, mens `no_grad()`
hindrer gradientregistrering. Ingen av dem oppdaterer parametere.


## Feilsøking i par – fem minutter

Velg to symptomer. Beskriv første ting dere vil undersøke før dere endrer koden:

- `mat1 and mat2 shapes cannot be multiplied`.
- Feilmelding om at etikettene må være `Long`.
- Tapet endres nesten ikke etter fem epoker.
- Testresultatet endres når man glemmer å kjøre modellcellen på nytt.

Sammenlign forklaringen med en annen gruppe.


**Mitt svar:**

Ved matriseformfeil skriver jeg ut formene før og etter `Flatten` og sjekker
784 innganger. Ved etikettfeil sjekker jeg `[B]` og `torch.int64`.
Ved uendret tap undersøker jeg at `backward()` kommer før `step()`, og at
optimizer tilhører modellen som faktisk brukes. Ved gjentatt kjøring undersøker
jeg om modellen er opprettet på nytt eller trent videre; initialisering og stokking varierer også.


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Start her:** [PyTorch – Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html). Sammenlign `train_loop` og `test_loop` med dine egne løkker. Finn linjene som oppdaterer parametere, og forklar hvorfor de bare finnes i treningen.
- **Samlet eksempel:** [PyTorch – Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html). Viser data, modell, trening og evaluering i sammenheng. Eksemplet bruker FashionMNIST og en annen modell; bruk det til å kjenne igjen mønsteret. Lagring av modellen er valgfritt viderearbeid.
